# OCONUS NGWPC Hydrofabric Demo
This notebook walks through the following PI-9 acceptance criteria for the OCONUS (Alaska, Hawaii, Puerto Rico/Virgin Islands) domains. This should be run after building OCONUS NHF.

- domains include PRVI, Hawaii, and Alaska
- existence of the same river miles covered in the operational version of the NWM
- ensuring rivers can be represented as a directed acyclic graphs for routing
- connectivity checks
- flowpath and divide statistics (drainage area, length, etc)
- Ensure that hydrofabric fully complies with the HY_Features (WaterML2 Part 3) data model standard as extended to include flowlines that reduce artificial trans-basin transfers of flow.
- POIs should include at minimum all existing operational data assimilation gages as well as existing calibration gages and waterbodies (i.e. lake/reservoir) and their outlet nexuses.
- Every attempt will be made to maximize the NGWPC Hydrofabric such that the number of divides between 3-10 sq. km and verify that routing computational unit lengths (derived from flowpaths and flowlines) are an integer multiple of a 300 m discretization (acceptable range: 250-350 m)

In [ ]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

In [ ]:
path_ak = Path("../data/ak_nhf_1.1.3.gpkg")
path_hi = Path("../data/hi_nhf_1.1.3.gpkg")
path_prvi =  Path("../data/prvi_nhf_1.1.3.gpkg")
path_nhf = Path("../data/nhf_1.1.3.gpkg")
path_nwm_v2_1 = Path("../data/nwm_v2_1_hydrofabric.gdb")

km_to_miles = 0.6213712

prvi_crs = 6566
hi_crs = 32604
conus_crs = 5070

pd.set_option("display.max_columns", None)

## Existence of the same river miles covered in the operational version of the NWM

Virtual flowpaths is the routing layer that includes all flowpaths. The sum of virtual flowpath is the sum of all NHF river miles.

In [ ]:
gdf_ak_vfp = gpd.read_file(path_ak, layer="virtual_flowpaths")
gdf_hi_vfp = gpd.read_file(path_hi, layer="virtual_flowpaths")
gdf_prvi_vfp = gpd.read_file(path_prvi, layer="virtual_flowpaths")
gdf_sconus_vfp = gpd.read_file(path_nhf, layer="virtual_flowpaths")

gdf_prvi_v2_1 = gpd.read_file(path_nwm_v2_1, layer="nwm_reaches_puertorico")
gdf_hi_v2_1 = gpd.read_file(path_nwm_v2_1, layer="nwm_reaches_hawaii")
gdf_conus_v2_1 = gpd.read_file(path_nwm_v2_1, layer="nwm_reaches_conus")

In [ ]:
# Calculate virtual flowpath length in miles
lengths = {}
for k, v in dict(zip(["AK", "HI", "PRVI", "CONUS"], [gdf_ak_vfp, gdf_hi_vfp, gdf_prvi_vfp, gdf_sconus_vfp], strict=False)).items():
    lengths[k] = int(v["length_km"].sum() * km_to_miles)

# Puerto Rico miles in HF 2.1 / NWM v3
gdf_prvi_v2_1 = gdf_prvi_v2_1.to_crs(prvi_crs)
prvi_2_1_miles = int(gdf_prvi_v2_1.geometry.length.sum() / 1000 * km_to_miles)

# Hawaii miles in HF 2.1 / NWM v3
gdf_hi_v2_1 = gdf_hi_v2_1.to_crs(hi_crs)
hi_2_1_miles = int(gdf_hi_v2_1.geometry.length.sum() / 1000 * km_to_miles)

# Hawaii miles in HF 2.1 / NWM v3
gdf_conus_v2_1 = gdf_conus_v2_1.to_crs(conus_crs)
conus_2_1_miles = int(gdf_conus_v2_1.geometry.length.sum() / 1000 * km_to_miles)

# merge dataframes
df_lengths = pd.DataFrame(data={"domain":list(lengths.keys()), "river_miles":list(lengths.values())})
df_lengths = df_lengths.merge(pd.DataFrame(data={"domain":["PRVI", "HI", "CONUS"], "nwm_v3_miles":[prvi_2_1_miles, hi_2_1_miles, conus_2_1_miles]}, dtype=(str, int)), on ="domain", how="left")

display(df_lengths)
print(f"Sum River Miles: {df_lengths["river_miles"].sum()}")


## Ensuring rivers can be represented as a directed acyclic graphs for routing / connectivity checks

TODO @Dylan: DAGs

## Flowpath and divide statistics (drainage area, length, etc)
Show the attributes available for divides and flowpath layers.

In [ ]:
def show_attributes(domain_path: Path, domain_name: str):
    """Display the fields in flowpaths and divides"""
    gdf_fp = gpd.read_file(domain_path, layer="flowpaths")
    print(f"{domain_name} Flowpath attributes")
    display(pd.DataFrame(data={"Columns":gdf_fp.columns}))

    gdf_div = gpd.read_file(domain_path, layer="divides")
    print(f"{domain_name} Divide Attributes")
    display(pd.DataFrame(data={"Columns":gdf_div.columns}))

In [ ]:
# AK
show_attributes(path_ak, domain_name="AK")

In [ ]:
# HI
show_attributes(path_hi, domain_name="HI")

In [ ]:
# PRVI
show_attributes(path_hi, domain_name="PRVI")

## Ensure that hydrofabric fully complies with the HY_Features (WaterML2 Part 3) data model standard as extended to include flowlines that reduce artificial trans-basin transfers of flow.
TODO @DYLAN Add text and go to OE link

## POIs should include at minimum all existing operational data assimilation gages as well as existing calibration gages and waterbodies (i.e. lake/reservoir) and their outlet nexuses

### Waterbodies / Lakes
The `lakes` layer was built in NHF to be a 1:1 representation of NWM operational waterbodies. The `lakes` layer retains all data from Hydrofabric 2.2 (Puerto Rico and Hawaii) and LAKEPARM (Alaska).

Where polygons were available (HI and PRVI), lakes were mapped to the most downstream intersecting flowpath. The most downstream flowpath is chosen with the minimum hydrosequence. In Alaska, points were mapped to their nearest flowpath.

In [ ]:
def compare_lakes(nhf_path: Path, nwm_path: Path, domain: str, id_field: str):
    """Compare if lakes are present in an NWM source file and an NHF gages layer"""
    gdf_nwm = gpd.read_file(nwm_path)
    gdf_nhf = gpd.read_file(nhf_path, layer="lakes")
    print(f"{domain} NHF lakes: {len(gdf_nhf)}")
    print(f"{domain} NWM lakes: {len(gdf_nwm)}")
    print(f"{domain} lakes COMID in NWM: {len(gdf_nhf.loc[gdf_nhf['lake_id'].isin(gdf_nwm[id_field])])}")
    display(gdf_nhf.head())

Alaska lakes were retrieved from [NWM v3.0.18 LAKEPARM_AK.nc](https://www.nco.ncep.noaa.gov/pmb/codes/nwprod/nwm.v3.0.18/parm/domain_alaska/LAKEPARM_AK.nc) and saved to a GPKG.

In [ ]:
compare_lakes(path_ak, "../data/lakes/input/ak_lakeparm.gpkg", "AK", "lake_id" )

Hawaii lakes were extracted from `nwm_lakes.gpkg` provided by OWP in January 2026. The file was clipped to Hawaii state borders. `lake_id` and `newID` are both NHD COMID.

In [ ]:
compare_lakes(path_hi, "../data/lakes/input/nwm_lakes_hi_input.gpkg", "HI", "newID")

Puerto Rico / Virgin Islands lakes were extracted from `nwm_lakes.gpkg` provided by OWP in January 2026. The file was clipped to Puerto Rico/Virgin Islands borders. `lake_id` and `newID` are both NHD COMID.

In [ ]:
compare_lakes(path_prvi, "../data/lakes/input/nwm_lakes_prvi_input.gpkg", "PRVI", "newID")

### Gages
Gages were extracted from routelink and USGS. Routelink files were downloaded from NWM v3.0.18, converted to GPKG in EPSG:4326, and extracted any row with a populated gage ID field. 

If upstream area information was available, gages were matched to flowpaths/divides using it. If it was not available, gages were matched to nearest flowpath. See connectivity columns in tables below (fp_id, virtual_fp_id, dn_nex_id, dn_virtual_nex_id).

In [ ]:
def compare_gages(nhf_path: Path, routelink_path: Path, domain: str):
    """Compare if gages are present in routelink and an NHF gages layer"""
    gdf_routelink = gpd.read_file(routelink_path)

    # extract rows with gages
    gdf_routelink["gages"] = gdf_routelink["gages"].str.strip()
    gdf_routelink = gdf_routelink.loc[gdf_routelink["gages"] != ""].copy()

    gdf_gages = gpd.read_file(nhf_path, layer="gages")
    print(f"{domain} NHF gages: {len(gdf_gages)}")
    print(f"{domain} Routelink gages: {len(gdf_routelink)}")
    print(f"{domain} gage ID in Routelink: {len(gdf_gages.loc[gdf_gages['site_no'].isin(gdf_routelink['gages'])])}")
    display(gdf_gages.head())

In [ ]:
# Function used in NHF-builds to extract routelink gages - this is for demonstration purposes only
def append_from_routelink(
    gdf: gpd.GeoDataFrame, routelink: Path, id_col_name: str, shape: Path | None
) -> gpd.GeoDataFrame:
    """Append gages from RouteLink file to GeoDataFrame

    Use ogr2ogr to convert NC file to GPKG and add EPSG:4326 georef i.e. ogr2ogr RouteLink.gpkg RouteLink.nc -t_srs EPSG:4326 -s_srs EPSG:4326

    Parameters
    ----------
    gdf: GeoDataFrame
        Input dataframe to append to
    routelink : Path
        RouteLink file to extract from
    id_col_name: str
        Column to pull from for site_no in RouteLink
    shape: Path | None
        Shapefile to use for clipping
    """
    gages = gpd.read_file(routelink).to_crs(gdf.crs)

    # first get gages only
    gages = gages.loc[gages[id_col_name].str.strip() != ""].copy()

    # then check intersection if requested
    if shape:
        # Get boundary to clip to
        shp = gpd.read_file(shape).to_crs(gdf.crs)
        merged_geom = shp["geometry"].union_all()
        gages = gages.loc[gages["geometry"].intersects(merged_geom), :].copy()

    gages = gages.rename(columns={id_col_name: "site_no"})
    gages["site_no"] = gages["site_no"].str.strip()

    gages = gpd.GeoDataFrame(gages[["geometry", "site_no"]][~gages["site_no"].isin(gdf["site_no"])].copy())
    # logger.info(f"gages: added {len(gages)} gages from RouteLink not already present in dataset") # commetned for missing imports in demonstration
    gages["status"] = "routelink"
    gages = pd.concat([gdf, gages])
    gages["geometry"] = gages["geometry"].force_2d()

    return gages

**Alaska Note**: There are some missing gages in Alaska due to a domain mismatch between the NHF AK reference and the NWM AK domain. The GEOGLOWS dataset used for the reference does not include some coastal areas of AK. This domain problem will be rectified in the next version of AK domain.

In [ ]:
# AK
compare_gages(path_ak, Path("../data/gages/routelink/RouteLink_AK_EPSG4326.gpkg"), "AK")

In [ ]:
# HI
compare_gages(path_hi, Path("../data/gages/routelink/RouteLink_HI_EPSG4326.gpkg"), "Hawaii")

In [ ]:
# PRVI
compare_gages(path_prvi, Path("../data/gages/routelink/RouteLink_PRVI_EPSG4326.gpkg"), "PRVI")

## Gage / Lake Flowpath Association
TODO @QUERCUS
demonstrate with figures how a point is matched to flowpath/nexus

## Every attempt will be made to maximize the NGWPC Hydrofabric such that the number of divides between 3-10 sq. km and verify that routing computational unit lengths (derived from flowpaths and flowlines) are an integer multiple of a 300 m discretization (acceptable range: 250-350 m)
TODO @DYLAN